# Notebook 4: Model Tournament and MLflow

**Five models (same experiment):**
1. **ALS** — RMSE on ratings (`engineered_features.parquet`, full scale).
2. **Logistic Regression** — accuracy / F1 on TF-IDF (`label` = rating ≥ 4).
3. **Random Forest** — same.
4. **Naive Bayes** — same.
5. **K-Means** — **silhouette** on the assembled `features` vector (user + product indices + TF-IDF).

NLP models use a **sample** of `amazon_clean` + saved `models/feature_pipeline` (tune `ML_SAMPLE_FRACTION`). **Metrics are not directly comparable** across model types; use MLflow to sort within each family.

**Run order:** load ALS data → train ALS → **then** run the **Prep: NLP & clustering** cell (so you are not caching ALS + NLP + K-Means together — avoids memory warnings).

Optional: `mlflow ui` from this directory to compare runs.

In [21]:
import os
import mlflow
from pyspark.ml import PipelineModel
from pyspark.ml.recommendation import ALS
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, NaiveBayes
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    RegressionEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator,
)
from pyspark.sql.functions import col, when

In [22]:
import os, subprocess
os.environ["JAVA_HOME"] = subprocess.check_output(
    ["/usr/libexec/java_home", "-v", "17"]
).decode().strip()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Amazon_Model_Tournament") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
print("Spark", spark.version, "| UI: http://localhost:4040")

Spark 4.1.1 | UI: http://localhost:4040


In [23]:
# Local tracking store (works offline). Same runs appear in `mlflow ui`.
_mlruns = os.path.abspath(os.path.join(os.getcwd(), "mlruns"))
mlflow.set_tracking_uri("file:" + _mlruns)
mlflow.set_experiment("Amazon_Product_Recommendation")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:/Users/bavishna/Documents/Bavishna_Masters/Sem_3/DATA228/Project/Final_project/mlruns


In [24]:
# ---- ALS only (full engineered Parquet). NLP/K-Means prep runs in a later cell after ALS finishes.
feat_path = "data/engineered_features.parquet"
abs_path = "file:///" + os.path.abspath(feat_path).replace("\\", "/")
als_all = spark.read.parquet(abs_path)
need = {"user_id_index", "product_id_index", "rating"}
missing = need - set(als_all.columns)
if missing:
    raise ValueError(f"engineered_features missing columns: {missing}. Re-run Notebook 3.")

als_all = als_all.select(
    col("user_id_index").cast("float"),
    col("product_id_index").cast("float"),
    col("rating").cast("float"),
).na.drop(subset=["user_id_index", "product_id_index", "rating"])

(train_als, test_als) = als_all.randomSplit([0.8, 0.2], seed=42)
train_als = train_als.cache()
print("ALS train:", f"{train_als.count():,}", "| test:", f"{test_als.count():,}")

26/05/06 13:56:18 WARN CacheManager: Asked to cache already cached data.        
26/05/06 13:56:18 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:56:19 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB


ALS train: 5,027,535 | test: 1,257,200


26/05/06 13:56:26 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:56:55 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:56:58 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 13:57:14 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB


Classification train: 603,827 | K-Means train: 603,752 (sample=0.12)


## 1) ALS (collaborative filtering)

In [25]:
with mlflow.start_run(run_name="ALS_Collaborative_Filtering"):
    als = ALS(
        maxIter=10,
        regParam=0.1,
        rank=10,
        userCol="user_id_index",
        itemCol="product_id_index",
        ratingCol="rating",
        coldStartStrategy="drop",
    )
    model = als.fit(train_als)
    predictions = model.transform(test_als)
    evaluator = RegressionEvaluator(
        metricName="rmse", labelCol="rating", predictionCol="prediction"
    )
    rmse = evaluator.evaluate(predictions)
    mlflow.log_param("rank", als.getRank())
    mlflow.log_param("regParam", als.getRegParam())
    mlflow.log_param("maxIter", als.getMaxIter())
    mlflow.log_metric("rmse", float(rmse))
    mlflow.spark.log_model(model, "als_model")
    os.makedirs("models", exist_ok=True)
    model.write().overwrite().save("models/als_recommendation_model")

train_als.unpersist()
print(f"ALS RMSE (test): {rmse:.4f}")

26/05/06 13:57:51 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:57:52 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:57:53 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:57:54 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:57:56 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:57:57 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:57:59 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:58:00 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:58:01 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:58:02 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:58:04 WARN DAGScheduler: Broadcasting large task binary with size 30.3 MiB
26/05/06 13:58:05 WARN DAGScheduler: Broadc

ALS RMSE (test): 2.0782


## Prep: NLP & clustering (run **after** the ALS cell)

The next cell samples `amazon_clean`, runs `models/feature_pipeline`, and caches `train_cls` / `train_km`. Keeping this **separate** from the ALS load avoids caching three huge datasets at once (fewer `MemoryStore` warnings).

In [ ]:
# LR / RF / NB / K-Means: sample cleaned reviews + saved NLP pipeline (increase fraction toward 1.0 = slower)
ML_SAMPLE_FRACTION = 0.12

clean_path = "data/amazon_clean.parquet/clean_data.parquet"
clean_abs = "file:///" + os.path.abspath(clean_path).replace("\\", "/")
raw_df = spark.read.parquet(clean_abs)
if ML_SAMPLE_FRACTION < 1.0:
    raw_df = raw_df.sample(withReplacement=False, fraction=ML_SAMPLE_FRACTION, seed=42)

pipe_path = os.path.abspath("models/feature_pipeline")
if not os.path.isdir(pipe_path):
    raise FileNotFoundError(f"Missing {pipe_path}. Run Notebook 3 first.")
pipeline_model = PipelineModel.load(pipe_path)
feat_df = pipeline_model.transform(raw_df)
feat_df = feat_df.withColumn(
    "label", when(col("rating") >= 4.0, 1.0).otherwise(0.0)
)

cls_all = feat_df.select("tfidf_features", "label").na.drop()
(train_cls, test_cls) = cls_all.randomSplit([0.8, 0.2], seed=43)
train_cls = train_cls.cache()

km_all = feat_df.select("features").na.drop()
(train_km, test_km) = km_all.randomSplit([0.8, 0.2], seed=44)
train_km = train_km.cache()

print(
    "Classification train:", f"{train_cls.count():,}",
    "| K-Means train:", f"{train_km.count():,}",
    f"(sample={ML_SAMPLE_FRACTION})",
)

## 2–4) Classification on TF-IDF (binary label: rating ≥ 4)

Metrics: **accuracy**, **f1** (compare these runs in MLflow; not comparable to ALS RMSE).

In [26]:
acc_eval = MulticlassClassificationEvaluator(
    predictionCol="prediction", labelCol="label", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    predictionCol="prediction", labelCol="label", metricName="f1"
)

with mlflow.start_run(run_name="LogisticRegression_TFIDF"):
    lr = LogisticRegression(
        featuresCol="tfidf_features", labelCol="label", maxIter=40, family="multinomial"
    )
    lr_model = lr.fit(train_cls)
    lr_pred = lr_model.transform(test_cls)
    mlflow.log_metric("accuracy", float(acc_eval.evaluate(lr_pred)))
    mlflow.log_metric("f1", float(f1_eval.evaluate(lr_pred)))
    mlflow.log_param("maxIter", 40)
    mlflow.spark.log_model(lr_model, "lr_model")

with mlflow.start_run(run_name="RandomForest_TFIDF"):
    rf = RandomForestClassifier(
        featuresCol="tfidf_features",
        labelCol="label",
        numTrees=40,
        maxDepth=12,
        seed=42,
    )
    rf_model = rf.fit(train_cls)
    rf_pred = rf_model.transform(test_cls)
    mlflow.log_metric("accuracy", float(acc_eval.evaluate(rf_pred)))
    mlflow.log_metric("f1", float(f1_eval.evaluate(rf_pred)))
    mlflow.log_param("numTrees", 40)
    mlflow.log_param("maxDepth", 12)
    mlflow.spark.log_model(rf_model, "rf_model")

with mlflow.start_run(run_name="NaiveBayes_TFIDF"):
    nb = NaiveBayes(
        featuresCol="tfidf_features", labelCol="label", smoothing=1.0, modelType="multinomial"
    )
    nb_model = nb.fit(train_cls)
    nb_pred = nb_model.transform(test_cls)
    mlflow.log_metric("accuracy", float(acc_eval.evaluate(nb_pred)))
    mlflow.log_metric("f1", float(f1_eval.evaluate(nb_pred)))
    mlflow.log_param("smoothing", 1.0)
    mlflow.spark.log_model(nb_model, "nb_model")

train_cls.unpersist()
print("Logged LR, RF, NB (accuracy + f1).")

26/05/06 13:59:11 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:14 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:15 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:17 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:19 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:20 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:21 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:23 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:24 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:26 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:28 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB
26/05/06 13:59:29 WARN DAGScheduler: Broadc

[1417.472s][warning][gc,alloc] Executor task launch worker for task 9.0 in stage 958.0 (TID 2491): Retried waiting for GCLocker too often allocating 68828 words


26/05/06 14:04:09 WARN MemoryStore: Not enough space to cache rdd_1636_0 in memory! (computed 55.1 MiB so far)
26/05/06 14:04:09 WARN MemoryStore: Not enough space to cache rdd_1636_2 in memory! (computed 24.2 MiB so far)
26/05/06 14:04:09 WARN MemoryStore: Not enough space to cache rdd_1636_5 in memory! (computed 55.1 MiB so far)
26/05/06 14:04:09 WARN MemoryStore: Not enough space to cache rdd_1636_6 in memory! (computed 129.9 MiB so far)
26/05/06 14:04:09 WARN MemoryStore: Not enough space to cache rdd_1636_7 in memory! (computed 129.9 MiB so far)
26/05/06 14:04:14 WARN DAGScheduler: Broadcasting large task binary with size 63.9 MiB
26/05/06 14:04:16 WARN MemoryStore: Not enough space to cache rdd_1636_7 in memory! (computed 55.1 MiB so far)
26/05/06 14:04:16 WARN MemoryStore: Not enough space to cache rdd_1636_4 in memory! (computed 4.3 MiB so far)
26/05/06 14:04:16 WARN MemoryStore: Not enough space to cache rdd_1636_0 in memory! (computed 2.5 MiB so far)
26/05/06 14:04:16 WARN Me

[1431.655s][warning][gc,alloc] Executor task launch worker for task 3.0 in stage 962.0 (TID 2529): Retried waiting for GCLocker too often allocating 137875 words


26/05/06 14:04:23 WARN MemoryStore: Not enough space to cache rdd_1636_4 in memory! (computed 129.9 MiB so far)
26/05/06 14:04:23 WARN MemoryStore: Not enough space to cache rdd_1636_3 in memory! (computed 36.5 MiB so far)
26/05/06 14:04:28 WARN DAGScheduler: Broadcasting large task binary with size 64.8 MiB
26/05/06 14:04:30 WARN MemoryStore: Not enough space to cache rdd_1636_6 in memory! (computed 24.2 MiB so far)
26/05/06 14:04:30 WARN MemoryStore: Not enough space to cache rdd_1636_4 in memory! (computed 36.5 MiB so far)
26/05/06 14:04:30 WARN MemoryStore: Not enough space to cache rdd_1636_2 in memory! (computed 24.2 MiB so far)
26/05/06 14:04:30 WARN MemoryStore: Not enough space to cache rdd_1636_1 in memory! (computed 36.5 MiB so far)
26/05/06 14:04:30 WARN MemoryStore: Not enough space to cache rdd_1636_7 in memory! (computed 55.1 MiB so far)
26/05/06 14:04:30 WARN MemoryStore: Not enough space to cache rdd_1636_3 in memory! (computed 36.5 MiB so far)
26/05/06 14:04:30 WARN M

Logged LR, RF, NB (accuracy + f1).


## 5) K-Means (assembled `features` vector)

Metric: **silhouette** on the held-out split (higher is better, roughly [-1, 1]).

In [27]:
km_eval = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

with mlflow.start_run(run_name="KMeans_features"):
    kmeans = KMeans(featuresCol="features", k=5, seed=42, maxIter=30)
    km_model = kmeans.fit(train_km)
    km_pred = km_model.transform(test_km)
    silhouette = float(km_eval.evaluate(km_pred))
    mlflow.log_metric("silhouette", silhouette)
    mlflow.log_param("k", 5)
    mlflow.log_param("maxIter", 30)
    mlflow.spark.log_model(km_model, "kmeans_model")

train_km.unpersist()
print(f"K-Means silhouette (test): {silhouette:.4f}")

26/05/06 14:08:15 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:18 WARN DAGScheduler: Broadcasting large task binary with size 30.9 MiB
26/05/06 14:08:19 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:22 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:25 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:28 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:30 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:33 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:35 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:37 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:39 WARN DAGScheduler: Broadcasting large task binary with size 63.3 MiB
26/05/06 14:08:41 WARN DAGScheduler: Broadc

K-Means silhouette (test): 0.6675
